In [1]:
import json
from pathlib import Path
from string import ascii_uppercase

import numpy as np
import scipy.io as sio
import torch

from csi_vae_gumbel.dataset import CSIDataset
from csi_vae_gumbel.models.vae import Parameters
from csi_vae_gumbel.models.vae.single_antenna_vae import SingleAntennaVAE
from csi_vae_gumbel.settings import Settings

settings = Settings()

GPU_ID = 1

/mnt/servicesdata/lcotti/csi-vae-gumbel/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_best_model() -> tuple[SingleAntennaVAE, Parameters]:
    """Load the best model and its parameters from the study results."""
    with Path(f"../{settings.study_path}/study_results.json").open("r") as f:
        study_info = json.load(f)

    best_model_path = Path(f"../{settings.study_path}/trial_{study_info['best_trial'][0]}")

    with Path(best_model_path / "results.json").open("r") as f:
        info = json.load(f)

    params = Parameters(
        final_cap=info["final_cap"],
        start_gumbel_temp=info["start_gumbel_temp"],
        final_kl_weight=info["final_kl_weight"],
        latent_dim=info["latent_dim"],
    )

    vae_model = SingleAntennaVAE(
        settings.train_window_size,
        settings.n_subcarriers,
        settings.n_categories,
        params.latent_dim,
    )
    best_model_weights = torch.load(best_model_path / "model.pt")
    vae_model.load_state_dict(best_model_weights)

    return vae_model, params

vae, params = get_best_model()
vae.eval()
vae = vae.to(GPU_ID)

In [3]:
files = [Path(f"../{settings.dataset_path}") / f"S1a_{x}.mat" for x in ascii_uppercase[: settings.n_activities]]
mats = [np.array(sio.loadmat(file)["csi"]) for file in files]

ds = CSIDataset(
    mats,
    settings.test_window_size,
    n_antennas=1,
    antenna_select=0,
    augment_probability=0,
)

dl = torch.utils.data.DataLoader(ds, batch_size=settings.train_batch_size, shuffle=False)

In [17]:
all_latents = []
all_labels = []

# 1. Convert to Categorical Indices first (T, latent_dim)
with torch.no_grad():
    for x, y in dl:
        batch_size = x.shape[0]
        window_size = x.shape[2] // settings.test_window_ratio

        x_r = x.view(batch_size * settings.test_window_ratio, x.shape[1], window_size, x.shape[3]).to(GPU_ID)

        _, z_hard, _ = vae(x_r)

        z_hard = z_hard.view(batch_size, settings.test_window_ratio, params.latent_dim, settings.n_categories)

        # Convert one-hot to index:
        # (B, test_window_ratio, latent_dim, n_categories) -> (B, test_window_ratio, latent_dim)
        z_hard = z_hard.argmax(dim=-1)

        # Concatenate the test_window_ratio dimension into the latent_dim dimension:
        # (B, test_window_ratio, latent_dim) -> (B, test_window_ratio * latent_dim)
        z_hard = z_hard.view(batch_size, -1)

        all_latents.append(z_hard.cpu())
        all_labels.append(y)

latents = torch.cat(all_latents, dim=0)
labels = torch.cat(all_labels, dim=0)

In [18]:
latents.shape

torch.Size([138612, 36])

In [19]:
output_path = Path(f"../{settings.study_path}/latents")
output_path.mkdir(parents=True, exist_ok=True)

np.save(output_path / "latents.npy", latents.numpy())
np.save(output_path / "labels.npy", labels.numpy())